# EmbeddingGemma 2 in practice

One model, one vector space, five kinds of input. This notebook covers the parts that matter when you actually build with it:

1. **Loading only the encoders you need:** text-only (270M) vs text+image vs text+audio vs everything (740M).
2. **Prompts:** queries and documents are embedded differently.
3. **Use case: search over a mixed media library.** Index photos, sound clips and notes once with the full model, then search with the text-only model.
4. **Matryoshka:** shrink vectors from 768 to 128 dims and check what you lose.
5. **Modality bias:** does a text query prefer text over the right photo?
6. **The search model on CPU.**
7. **Why not float16.**

Data: 12 photos from Wikimedia Commons, 10 sound clips from ESC-50, and 6 short notes. Run `get_data.py` first.

In [1]:
import os, time, json, gc
os.environ.setdefault("HF_HOME", "D:/hf-cache")

import numpy as np
import torch
from functools import lru_cache
from pathlib import Path
from sentence_transformers import SentenceTransformer

MODEL_ID = "google/embeddinggemma-2"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# bfloat16 or float32 only. The model card warns float16 gives NaN or silently worse vectors.
DTYPE = torch.bfloat16 if DEVICE == "cuda" and torch.cuda.is_bf16_supported() else torch.float32
DATA = Path("data")
print(DEVICE, DTYPE, torch.__version__, torch.cuda.get_device_name() if DEVICE == "cuda" else "")

cuda torch.bfloat16 2.14.1+cu130 NVIDIA GeForce RTX 4060 Laptop GPU


## 1. Loading only the encoders you need

EmbeddingGemma 2 is modular: a 270M text core, plus a 170M vision encoder and a 300M audio encoder that you can leave out.
You choose with `config_kwargs`: setting `vision_config` or `audio_config` to `None` means those weights are never loaded.

In [2]:
ENCODERS = {
    "text only":    {"vision_config": None, "audio_config": None},
    "text + image": {"audio_config": None},
    "text + audio": {"vision_config": None},
    "everything":   {},
}

def load(encoders: str, device: str = DEVICE, dtype=DTYPE) -> SentenceTransformer:
    return SentenceTransformer(
        MODEL_ID,
        device=device,
        model_kwargs={"torch_dtype": dtype},
        config_kwargs=ENCODERS[encoders],
    )

def release():
    # call after `del model` so the GPU memory is actually returned
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [3]:
rows = []
for name in ENCODERS:
    torch.cuda.reset_peak_memory_stats()
    t0 = time.perf_counter()
    m = load(name)
    load_s = time.perf_counter() - t0
    params = sum(p.numel() for p in m.parameters()) / 1e6
    gpu_mb = torch.cuda.memory_allocated() / 2**20
    rows.append((name, params, gpu_mb, load_s))
    del m
    release()

print(f"{'encoders':<14}{'params':>10}{'GPU memory':>14}{'load time':>12}")
for name, params, gpu_mb, load_s in rows:
    print(f"{name:<14}{params:>9.0f}M{gpu_mb:>11.0f} MB{load_s:>11.1f}s")


Loading weights:   0%|          | 0/413 [00:00<?, ?it/s]


Loading weights:   0%|          | 1/413 [00:00<01:03,  6.49it/s]


Loading weights:   5%|▍         | 20/413 [00:00<00:04, 94.41it/s]


Loading weights:  12%|█▏        | 51/413 [00:00<00:02, 170.43it/s]


Loading weights:  17%|█▋        | 70/413 [00:00<00:02, 169.34it/s]

...


Loading weights:  69%|██████▉   | 956/1376 [00:04<00:03, 129.33it/s]


Loading weights:  71%|███████▏  | 981/1376 [00:04<00:02, 141.47it/s]


Loading weights:  73%|███████▎  | 1002/1376 [00:04<00:02, 129.36it/s]


Loading weights: 100%|██████████| 1376/1376 [00:04<00:00, 311.88it/s]

encoders          params    GPU memory   load time
text only           271M        517 MB       13.5s
text + image        439M        849 MB       13.5s
text + audio        577M       1101 MB       21.6s
everything          744M       1432 MB       25.8s


## 2. Prompts: queries and documents are not the same

This is an asymmetric model. A search query is short and asks for something, while a document describes something.
The model was trained with a different instruction prefix for each side.

In [4]:
text_model = load("text only")
print("query prefix:   ", repr(text_model.prompts["SearchQuery"]))
print("document prefix:", repr(text_model.prompts["Document"]))


Loading weights:   0%|          | 0/413 [00:00<?, ?it/s]


Loading weights:   2%|▏         | 7/413 [00:00<00:06, 58.44it/s]


Loading weights:   3%|▎         | 13/413 [00:00<00:18, 21.11it/s]


Loading weights:   4%|▍         | 17/413 [00:00<00:17, 23.16it/s]


Loading weights: 100%|██████████| 413/413 [00:00<00:00, 539.74it/s]

query prefix:    'task: search result | query: '
document prefix: 'title: none | text: '


A small test: 12 questions and 12 answers. For each question, how far ahead of the closest wrong answer is the right one, with and without prompts?

In [5]:
PAIRS = [
    ("why is my pandas merge creating duplicate rows?",
     "If the join key is not unique in one of the tables, every match becomes its own row. Check df['key'].is_unique before merging."),
    ("why does my laptop fan get loud on video calls?",
     "Video calls encode the camera feed in real time, which keeps the CPU busy, so the fan speeds up to cool it."),
    ("why is my GPU running out of memory during training?",
     "Training keeps every layer's activations for the backward pass, so a bigger batch or longer sequences fill GPU memory fast."),
    ("how do I undo my last git commit but keep the changes?",
     "Run git reset --soft HEAD~1. The commit is removed, and your changes stay staged."),
    ("why is my Docker image so big?",
     "Each RUN line adds a layer, and build tools left in the final stage add hundreds of megabytes. Use a multi-stage build."),
    ("how long should I rest pizza dough?",
     "Let the dough rest in the fridge overnight. The slow rise gives it more flavour and makes it easier to stretch."),
    ("best time to visit Araku Valley",
     "October to March is pleasant in Araku, with cool mornings, and the coffee plantations are green after the monsoon."),
    ("how do I make filter coffee at home?",
     "Add three spoons of coffee powder to the upper chamber of the filter, pour boiling water, and wait 15 minutes for the decoction."),
    ("why do my curd batches turn sour quickly?",
     "Curd sours faster in hot weather. Set it in a cooler spot and move it to the fridge as soon as it sets."),
    ("what does a 401 error mean?",
     "A 401 means the server didn't get valid credentials. Check that your API key or token is sent in the Authorization header."),
    ("how do I speed up a slow SQL query?",
     "Look at the query plan first. A missing index on a column used in WHERE or JOIN is the most common cause."),
    ("why does my Python script use so much RAM?",
     "Reading a whole file into a list keeps every line in memory at once. Read it in chunks or use a generator instead."),
]
questions = [q for q, _ in PAIRS]
answers = [a for _, a in PAIRS]

for label, q_kwargs, a_kwargs in [
    ("with prompts", {"prompt_name": "SearchQuery"}, {"prompt_name": "Document"}),
    ("without",      {}, {}),
]:
    scores = text_model.similarity(text_model.encode(questions, **q_kwargs),
                                   text_model.encode(answers, **a_kwargs)).float().numpy()
    right = scores.diagonal()
    wrong = scores.copy()
    np.fill_diagonal(wrong, -1)
    closest_wrong = wrong.max(axis=1)
    unrelated = scores[~np.eye(len(PAIRS), dtype=bool)]
    top1 = (scores.argmax(axis=1) == np.arange(len(PAIRS))).mean()
    print(f"{label:<13} top-1 {top1:.0%} | right answer {right.mean():.3f} | closest wrong {closest_wrong.mean():.3f}"
          f" | lead {(right - closest_wrong).mean():.3f} | any unrelated pair {unrelated.mean():.3f}")
    for i in np.where(scores.argmax(axis=1) != np.arange(len(PAIRS)))[0]:
        print(f"{'':<13} missed {questions[i]!r} -> picked {answers[scores[i].argmax()][:45]!r}")

with prompts  top-1 92% | right answer 0.782 | closest wrong 0.646 | lead 0.135 | any unrelated pair 0.577
              missed 'why does my Python script use so much RAM?' -> picked "Training keeps every layer's activations for "


without       top-1 92% | right answer 0.805 | closest wrong 0.695 | lead 0.110 | any unrelated pair 0.630
              missed 'why does my Python script use so much RAM?' -> picked 'Each RUN line adds a layer, and build tools l'


## 3. Use case: search over a mixed media library

Photos, sound clips and notes in one place, the kind of mix you'd find in a shared drive or a team's archive. You type a query and get back whichever items match, in any format.

Two jobs with very different needs:

| Job | How often | Needs |
|---|---|---|
| **Indexing:** turn every photo, clip and note into a vector | Once per new file, as a background or batch job | Vision + audio encoders |
| **Searching:** turn the typed query into a vector | Every keystroke, must be fast | Text core only |

All modalities share one vector space, so a query embedded by the **text-only** model can be compared with photo and audio vectors made by the **full** model.

### 3a. Indexing (full model, runs once)

In [6]:
NOTES = {
    "note_vet":      "Vet appointment for Sheru on Friday at 5pm. Carry his vaccination card.",
    "note_pizza":    "Pizza dough: 500g maida, 325ml water, 10g salt, 3g yeast. Rest overnight in the fridge.",
    "note_trip":     "Vizag trip plan: two days at RK Beach, then a day in Araku Valley. Book the hotel on Beach Road.",
    "note_grocery":  "Grocery list: apples, curd, toor dal, onions, filter coffee powder.",
    "note_meeting":  "Standup: OCR pipeline still failing on scanned PDFs, Ravi to check the GPU quota, demo moved to Thursday.",
    "note_london":   "London trip with Amma: Big Ben, Westminster Bridge, a walk by the Thames, dinner in Southall.",
}

items = (
    [{"id": p.stem, "type": "image", "input": {"image": str(p)}} for p in sorted((DATA / "images").glob("*.jpg"))]
    + [{"id": p.stem, "type": "audio", "input": {"audio": str(p)}} for p in sorted((DATA / "audio").glob("*.wav"))]
    + [{"id": k, "type": "text", "input": v} for k, v in NOTES.items()]
)
print(len(items), "items:", {t: sum(i["type"] == t for i in items) for t in ("image", "audio", "text")})

28 items: {'image': 12, 'audio': 10, 'text': 6}


In [7]:
del text_model
release()
indexer = load("everything")

t0 = time.perf_counter()
vectors = indexer.encode([i["input"] for i in items], prompt_name="Document", batch_size=4)
print(f"indexed {len(items)} items in {time.perf_counter() - t0:.1f}s -> {vectors.shape}")

# Save the index, then unload the big model. Search never needs it again.
np.save("index_768.npy", vectors.astype(np.float32))
json.dump([{k: i[k] for k in ("id", "type")} for i in items], open("index_meta.json", "w"), indent=1)

# Keep one query vector from the full model, to compare with the text-only model later.
full_q = indexer.encode("a dog playing outside", prompt_name="SearchQuery")
del indexer
release()


Loading weights:   0%|          | 0/1376 [00:00<?, ?it/s]


Loading weights:   1%|          | 13/1376 [00:00<00:10, 126.38it/s]


Loading weights:   2%|▏         | 26/1376 [00:00<00:15, 88.84it/s] 


Loading weights:   3%|▎         | 39/1376 [00:00<00:13, 97.16it/s]


Loading weights:   4%|▎         | 50/1376 [00:00<00:19, 69.55it/s]

...


Loading weights:  45%|████▍     | 613/1376 [00:10<00:07, 100.78it/s]


Loading weights:  45%|████▌     | 625/1376 [00:10<00:13, 55.67it/s] 


Loading weights:  46%|████▌     | 636/1376 [00:10<00:13, 53.96it/s]


Loading weights: 100%|██████████| 1376/1376 [00:10<00:00, 125.76it/s]

indexed 28 items in 6.9s -> (28, 768)


### 3b. Searching (text-only model)

In [8]:
searcher = load("text only")
index = np.load("index_768.npy")
meta = json.load(open("index_meta.json"))

# Same query, text-only model. Is it the same vector the full model produced?
lite_q = searcher.encode("a dog playing outside", prompt_name="SearchQuery")
print("cosine(full, text-only) =", float(np.dot(full_q, lite_q) / np.linalg.norm(full_q) / np.linalg.norm(lite_q)))
print("max abs difference      =", float(np.abs(full_q - lite_q).max()))


Loading weights:   0%|          | 0/413 [00:00<?, ?it/s]


Loading weights:  10%|█         | 43/413 [00:00<00:01, 264.41it/s]


Loading weights:  17%|█▋        | 70/413 [00:00<00:01, 208.74it/s]


Loading weights:  22%|██▏       | 91/413 [00:00<00:01, 186.39it/s]


Loading weights: 100%|██████████| 413/413 [00:00<00:00, 793.20it/s]

cosine(full, text-only) = 1.0000001192092896
max abs difference      = 0.0


In [9]:
@lru_cache(maxsize=None)
def embed_query(query: str) -> np.ndarray:
    return searcher.encode(query, prompt_name="SearchQuery")

def search(query: str, k: int = 3, dim: int = 768, index=index):
    q = embed_query(query)[:dim]
    docs = index[:, :dim]
    # Matryoshka: after cutting dimensions, re-normalise before cosine similarity
    q = q / np.linalg.norm(q)
    docs = docs / np.linalg.norm(docs, axis=1, keepdims=True)
    scores = docs @ q
    top = np.argsort(-scores)[:k]
    return [(meta[i]["type"], meta[i]["id"], float(scores[i])) for i in top]

QUERIES = [
    "a dog",
    "waves crashing on the shore",
    "a baby",
    "bells ringing in London",
    "a rooster crowing at dawn",
    "something to eat",
    "snowy mountain peak",
    "someone typing on a computer",
    "emergency vehicle",
    "a rainy day",
]

t0 = time.perf_counter()
for q in QUERIES:
    hits = search(q)
    print(f"{q!r:34}", " | ".join(f"{t:5} {i:<13} {s:.2f}" for t, i, s in hits))
print(f"\n{(time.perf_counter() - t0) / len(QUERIES) * 1000:.0f} ms per query (embed + search)")

'a dog'                            image labrador      0.75 | audio dog           0.75 | audio rooster       0.74


'waves crashing on the shore'      image beach         0.78 | audio sea_waves     0.76 | audio train         0.74
'a baby'                           image baby          0.78 | audio rooster       0.74 | audio crying_baby   0.74


'bells ringing in London'          audio church_bells  0.76 | audio clock_tick    0.74 | image big_ben       0.72


'a rooster crowing at dawn'        image rooster       0.75 | audio rooster       0.72 | audio dog           0.70
'something to eat'                 image pizza         0.72 | audio keyboard_typing 0.69 | image apple         0.69
'snowy mountain peak'              image everest       0.78 | audio rain          0.69 | audio sea_waves     0.69
'someone typing on a computer'     audio keyboard_typing 0.76 | audio sea_waves     0.71 | audio train         0.70


'emergency vehicle'               

 audio rain          0.67 | audio train         0.67 | audio sea_waves     0.67
'a rainy day'                      audio sea_waves     0.73 | audio rain          0.72 | audio train         0.72

64 ms per query (embed + search)


Most results are right, but not all of them. Where did the clips that should have won actually land?

In [10]:
def rank_of(query: str, item_id: str) -> int:
    return next(r for r, (_, i, _) in enumerate(search(query, k=len(meta)), 1) if i == item_id)

for q, want in [("emergency vehicle", "siren"), ("a rainy day", "rain"), ("a dog", "dog")]:
    print(f"{q!r:22} -> '{want}' is rank {rank_of(q, want)} of {len(meta)}")

'emergency vehicle'    -> 'siren' is rank 5 of 28
'a rainy day'          -> 'rain' is rank 2 of 28
'a dog'                -> 'dog' is rank 2 of 28


Why do rain, sea waves and the train get mixed up? Compare the stored vectors directly:

In [11]:
unit = index / np.linalg.norm(index, axis=1, keepdims=True)
ids = [m["id"] for m in meta]
sim = lambda a, b: float(unit[ids.index(a)] @ unit[ids.index(b)])
for a, b in [("sea_waves", "train"), ("rain", "sea_waves"), ("rain", "train"), ("rain", "labrador")]:
    print(f"{a:>10} vs {b:<10} {sim(a, b):.2f}")

types = np.array([m["type"] for m in meta])
print()
for t in ("image", "audio", "text"):
    block = unit[types == t] @ unit[types == t].T
    print(f"average similarity between two {t} items: {block[~np.eye(len(block), dtype=bool)].mean():.2f}")

 sea_waves vs train      0.97
      rain vs sea_waves  0.93
      rain vs train      0.91
      rain vs labrador   0.65

average similarity between two image items: 0.76
average similarity between two audio items: 0.81
average similarity between two text items: 0.64


## 4. Matryoshka: 768 → 128 dimensions

The model is trained so the first 128, 256 or 512 numbers of each vector already make a usable embedding.
Two rules: cut from the front, and re-normalise afterwards (`search()` above does both).
How much ranking changes on our small library:

In [12]:
def top_ids(q, k, dim):
    # (type, id): the rooster photo and the rooster clip share the id "rooster"
    return [(t, i) for t, i, _ in search(q, k=k, dim=dim)]

print(f"{'dims':>5}{'bytes/vector':>14}{'top-1 same as 768':>20}{'top-3 overlap':>16}")
for dim in (768, 512, 256, 128):
    top1 = np.mean([top_ids(q, 1, dim) == top_ids(q, 1, 768) for q in QUERIES])
    top3 = np.mean([len(set(top_ids(q, 3, dim)) & set(top_ids(q, 3, 768))) / 3 for q in QUERIES])
    print(f"{dim:>5}{dim * 4:>14}{top1:>19.0%}{top3:>16.0%}")

 dims  bytes/vector   top-1 same as 768   top-3 overlap
  768          3072               100%            100%
  512          2048               100%             93%
  256          1024                90%             97%
  128           512                90%             87%


A common production pattern: search all vectors with 128 dims (cheap), then re-score only the top candidates with the full 768.

In [13]:
def search_two_stage(query: str, k: int = 3, shortlist: int = 10):
    q = embed_query(query)
    small = index[:, :128] / np.linalg.norm(index[:, :128], axis=1, keepdims=True)
    q_small = q[:128] / np.linalg.norm(q[:128])
    candidates = np.argsort(-(small @ q_small))[:shortlist]      # fast, coarse
    full = index[candidates] / np.linalg.norm(index[candidates], axis=1, keepdims=True)
    rescored = full @ (q / np.linalg.norm(q))                   # precise, few vectors
    order = candidates[np.argsort(-rescored)][:k]
    return [(meta[i]["type"], meta[i]["id"]) for i in order]

agree = np.mean([search_two_stage(q) == top_ids(q, 3, 768) for q in QUERIES])
print(f"two-stage (128 -> 768) matches full 768 top-3 exactly on {agree:.0%} of queries")

two-stage (128 -> 768) matches full 768 top-3 exactly on 100% of queries


## 5. Modality bias: does text win unfairly?

In a mixed index, a text query can favour text even when a photo is the better answer.
A quick check: five queries where both a note and a photo are relevant. Which one comes first, and by how much?

In [14]:
BIAS_QUERIES = {
    # query: the photo that should come up
    "a dog":                          "labrador",
    "pizza":                          "pizza",
    "a beach in Vizag":               "beach",
    "Big Ben":                        "big_ben",
    "apples":                         "apple",
}

for q, best in BIAS_QUERIES.items():
    hits = search(q, k=len(meta))
    rank = next(r for r, (_, i, _) in enumerate(hits, 1) if i == best)
    top_type, top_id, top_s = hits[0]
    best_s = next(s for _, i, s in hits if i == best)
    print(f"{q!r:26} top: {top_type:5} {top_id:<12} {top_s:.2f} | '{best}' rank {rank}, score {best_s:.2f}")

'a dog'                    top: image labrador     0.75 | 'labrador' rank 1, score 0.75


'pizza'                    top: image pizza        0.80 | 'pizza' rank 1, score 0.80


'a beach in Vizag'         top: text  note_trip    0.82 | 'beach' rank 2, score 0.77


'Big Ben'                  top: image big_ben      0.83 | 'big_ben' rank 1, score 0.83


'apples'                   top: image apple        0.79 | 'apple' rank 1, score 0.79


If text crowds out photos in your own data, a simple fix is to take the **top result from each modality** and merge them, so photos and clips always get a slot:

In [15]:
def search_per_modality(query: str, k_each: int = 1):
    hits = search(query, k=len(meta))
    out = []
    for t in ("image", "audio", "text"):
        out += [h for h in hits if h[0] == t][:k_each]
    return sorted(out, key=lambda h: -h[2])

for q in BIAS_QUERIES:
    print(f"{q!r:26}", " | ".join(f"{t:5} {i:<12} {s:.2f}" for t, i, s in search_per_modality(q)))

'a dog'                    image labrador     0.75 | audio dog          0.75 | text  note_vet     0.69
'pizza'                    image pizza        0.80 | text  note_pizza   0.76 | audio keyboard_typing 0.71
'a beach in Vizag'         text  note_trip    0.82 | image beach        0.77 | audio sea_waves    0.64
'Big Ben'                  image big_ben      0.83 | text  note_london  0.77 | audio church_bells 0.71
'apples'                   image apple        0.79 | text  note_grocery 0.70 | audio keyboard_typing 0.70


## 6. Bonus: the search model on CPU

The text-only model is small enough to run without a GPU. A rough check on this laptop's CPU (float32):

In [16]:
del searcher
release()
cpu_searcher = load("text only", device="cpu", dtype=torch.float32)
cpu_searcher.encode("warm up", prompt_name="SearchQuery")
t0 = time.perf_counter()
for q in QUERIES:
    cpu_searcher.encode(q, prompt_name="SearchQuery")
print(f"CPU: {(time.perf_counter() - t0) / len(QUERIES) * 1000:.0f} ms per query embedding")


Loading weights:   0%|          | 0/413 [00:00<?, ?it/s]


Loading weights:   9%|▊         | 36/413 [00:00<00:01, 287.35it/s]


Loading weights:  20%|██        | 84/413 [00:00<00:01, 292.60it/s]


Loading weights:  33%|███▎      | 135/413 [00:00<00:00, 284.86it/s]


Loading weights:  49%|████▊     | 201/413 [00:00<00:00, 389.31it/s]

...


Loading weights:  70%|███████   | 290/413 [00:00<00:00, 332.92it/s]


Loading weights:  84%|████████▍ | 348/413 [00:00<00:00, 393.75it/s]


Loading weights:  95%|█████████▍| 391/413 [00:01<00:00, 373.98it/s]


Loading weights: 100%|██████████| 413/413 [00:01<00:00, 355.35it/s]

CPU: 69 ms per query embedding


## 7. Why not float16?

The model card says float16 gives NaN or silently worse embeddings. Here's a short text and a long document, embedded in float32, bfloat16 and float16:

In [17]:
del cpu_searcher
release()

short_doc = "why is my pandas merge creating duplicate rows?"
long_doc = " ".join(list(NOTES.values()) * 50)

vecs = {}
for dtype in (torch.float32, torch.bfloat16, torch.float16):
    m = load("text only", dtype=dtype)
    vecs[dtype] = {"short": m.encode(short_doc, prompt_name="Document"),
                   "long": m.encode(long_doc, prompt_name="Document")}
    del m
    release()

for dtype in (torch.bfloat16, torch.float16):
    for name in ("short", "long"):
        v, ref = vecs[dtype][name].astype(np.float64), vecs[torch.float32][name].astype(np.float64)
        if np.isnan(v).any():
            print(f"{str(dtype):15} {name:<6} {int(np.isnan(v).sum())} of 768 values are NaN")
        else:
            print(f"{str(dtype):15} {name:<6} cosine to float32 = {np.dot(v, ref) / np.linalg.norm(v) / np.linalg.norm(ref):.5f}")


Loading weights:   0%|          | 0/413 [00:00<?, ?it/s]


Loading weights:  24%|██▍       | 101/413 [00:00<00:00, 958.25it/s]


Loading weights:  48%|████▊     | 197/413 [00:00<00:00, 907.61it/s]


Loading weights:  73%|███████▎  | 301/413 [00:00<00:00, 961.95it/s]


Loading weights:  99%|█████████▊| 407/413 [00:00<00:00, 946.95it/s]

...


Loading weights:  24%|██▍       | 101/413 [00:00<00:00, 1009.78it/s]


Loading weights:  53%|█████▎    | 220/413 [00:00<00:00, 1099.39it/s]


Loading weights:  80%|████████  | 331/413 [00:00<00:00, 1102.07it/s]


Loading weights: 100%|██████████| 413/413 [00:00<00:00, 1065.87it/s]

torch.bfloat16  short  cosine to float32 = 0.99996
torch.bfloat16  long   cosine to float32 = 0.99998
torch.float16   short  cosine to float32 = 1.00000
torch.float16   long   47 of 768 values are NaN


No error, no warning, just NaN for the long document. The reason is mean pooling: to get one vector, the model's token vectors are added up and then divided by the count. The sum is what overflows:

In [18]:
m = load("text only", dtype=torch.float32)
tokens = m.encode(long_doc, prompt_name="Document", output_value="token_embeddings")
print(f"tokens in the long document:           {tokens.shape[0]:,}")
print(f"largest value in one token vector:     {float(tokens.abs().max()):,.0f}")
print(f"largest value after adding all tokens: {float(tokens.sum(0).abs().max()):,.0f}")
print(f"largest number float16 can store:      {torch.finfo(torch.float16).max:,.0f}")
del m
release()


Loading weights:   0%|          | 0/413 [00:00<?, ?it/s]


Loading weights:  16%|█▌        | 66/413 [00:00<00:00, 647.68it/s]


Loading weights:  33%|███▎      | 136/413 [00:00<00:00, 663.05it/s]


Loading weights:  56%|█████▌    | 230/413 [00:00<00:00, 783.18it/s]


Loading weights:  79%|███████▉  | 326/413 [00:00<00:00, 836.90it/s]


Loading weights:  99%|█████████▉| 410/413 [00:00<00:00, 714.71it/s]


Loading weights: 100%|██████████| 413/413 [00:00<00:00, 727.75it/s]

tokens in the long document:           7,058
largest value in one token vector:     3,682
largest value after adding all tokens: 396,596
largest number float16 can store:      65,504
